# Bassac River Forecasting — Part 0: Data Cleaning & Calibration Check

Combines `01_clean_bassac.py`, `02_clean_kohkhel.py`, `04_clean_glofas.py`, and `06_check_calibration.py`.

In [1]:
# ==============================================================================
# 1. PATHS & IMPORTS
# ==============================================================================
from pathlib import Path
import numpy as np
import pandas as pd

ROOT_DIR = Path('..').resolve()
RAW_DIR = ROOT_DIR / 'data' / 'raw'
PROCESSED_DIR = ROOT_DIR / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
# ==============================================================================
# 2. CLEAN BASSAC_01 (15-MIN -> DAILY MEAN, REJECT <90 READINGS)
# ==============================================================================
raw_b = pd.read_csv(RAW_DIR / 'Bassac_01_ori.csv', low_memory=False)
raw_b['dt'] = pd.to_datetime(raw_b['Date'], format='%d/%m/%Y %H:%M', errors='coerce')
raw_b['WL'] = pd.to_numeric(raw_b['WL'], errors='coerce')
raw_b = raw_b.dropna(subset=['dt', 'WL']).sort_values('dt').drop_duplicates(subset=['dt'])

idx_b = raw_b.set_index('dt')
daily_mean = idx_b['WL'].resample('D').mean()
daily_count = idx_b['WL'].resample('D').count()

b_0700 = raw_b[raw_b['dt'].dt.time == pd.to_datetime('07:00').time()].copy()
b_0700['day'] = b_0700['dt'].dt.normalize()
s_0700 = b_0700.set_index('day')['WL']

bassac_daily = pd.DataFrame({
    'date': daily_mean.index.strftime('%Y-%m-%d'),
    'WL': daily_mean.values,
    'WL_0700': daily_mean.index.map(s_0700).values,
    'n_readings': daily_count.values,
})
bassac_daily.loc[bassac_daily['n_readings'] < 90, 'WL'] = np.nan
print('bassac_daily:', bassac_daily.shape, '| Valid WL:', bassac_daily['WL'].notna().sum())

bassac_daily.to_csv(PROCESSED_DIR / 'bassac_daily.csv', index=False)

bassac_daily: (1438, 4) | Valid WL: 1327


In [3]:
# ==============================================================================
# 3. CLEAN DHRW KOH_KHEL & GLOFAS UPSTREAM DISCHARGE
# ==============================================================================
raw_k = pd.read_csv(RAW_DIR / 'dhrw_water_level_2020_2026.csv', low_memory=False)
kohkhel_daily = pd.DataFrame({
    'date': pd.to_datetime(raw_k['date'], errors='coerce'),
    'WL_raw': pd.to_numeric(raw_k['Koh_Khel'], errors='coerce'),
}).dropna().sort_values('date').drop_duplicates(subset=['date']).reset_index(drop=True)
kohkhel_daily['date'] = kohkhel_daily['date'].dt.strftime('%Y-%m-%d')

raw_g = pd.read_csv(RAW_DIR / 'glofas_discharge_phnompenh.csv', low_memory=False)
glofas_daily = pd.DataFrame({
    'date': pd.to_datetime(raw_g['date'], errors='coerce'),
    'upstream_discharge': pd.to_numeric(raw_g['Q'], errors='coerce'),
}).dropna().sort_values('date').drop_duplicates(subset=['date']).reset_index(drop=True)
glofas_daily['date'] = glofas_daily['date'].dt.strftime('%Y-%m-%d')

print('kohkhel_daily:', kohkhel_daily.shape)
print('glofas_daily :', glofas_daily.shape)

kohkhel_daily.to_csv(PROCESSED_DIR / 'kohkhel_daily.csv', index=False)
glofas_daily.to_csv(PROCESSED_DIR / 'glofas_daily.csv', index=False)

kohkhel_daily: (1498, 2)
glofas_daily : (3235, 2)


In [4]:
# ==============================================================================
# 4. VERIFY CALIBRATION FORMULA ON 2020-2023 OVERLAP (645 SHARED DAYS)
# ==============================================================================
shared = bassac_daily.dropna(subset=['WL']).merge(kohkhel_daily, on='date', how='inner')
r = shared['WL'].corr(shared['WL_raw'])
slope, intercept = np.polyfit(shared['WL_raw'], shared['WL'], 1)

raw_mae = (shared['WL_raw'] - shared['WL']).abs().mean()
cal_mae = ((slope * shared['WL_raw'] + intercept) - shared['WL']).abs().mean()

print(f'Shared days : {len(shared)}')
print(f'Correlation : r = {r:.4f}')
print(f'Formula     : WL = {slope:.4f} * WL_raw + ({intercept:.4f})')
print(f'MAE before  : {raw_mae:.3f} m -> MAE after: {cal_mae:.3f} m')


Shared days : 643
Correlation : r = 0.9957
Formula     : WL = 0.7942 * WL_raw + (-0.4756)
MAE before  : 1.482 m -> MAE after: 0.093 m
